In [1]:
1+1

2

In [2]:
import os

In [4]:
%pwd

'c:\\Users\\godje\\OneDrive\\Escritorio\\Data-Science-Project-Stroke-Prediction-\\research'

In [5]:
os.chdir("../")

In [6]:
%pwd

'c:\\Users\\godje\\OneDrive\\Escritorio\\Data-Science-Project-Stroke-Prediction-'

In [45]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class ModelTrainerConfig:
    root_dir: Path
    train_data_path: Path
    test_data_path: Path
    model_name: str
    n_estimators: int
    learning_rate: float
    max_depth: int
    gamma: float
    subsample: float
    target_col: str


In [46]:
from src.stroke_prediction.constants import *
from src.stroke_prediction.utils.common import read_yaml, create_directories

In [51]:
class ConfiguartionManager:
    def __init__(self,
                 config_file_path = CONFIG_FILE_PATH,
                 params_file_path = PARAMS_FILE_PATH,
                 schema_file_path = SCHEMA_FILE_PATH): 
        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_file_path)
        self.schema = read_yaml(schema_file_path)

        create_directories([self.config.artifacts_root_dir])

    def get_model_trainer_config(self) -> ModelTrainerConfig:
        config = self.config.model_trainer
        params = self.params.XGBoost
        schema = self.schema.TARGET_COLUMN

        create_directories([config.root_dir])

        model_trainer_config = ModelTrainerConfig(
            root_dir = config.root_dir,
            train_data_path = config.train_data_path,
            test_data_path = config.test_data_path,
            model_name = config.model_name,
            n_estimators = params.n_estimators,
            learning_rate = params.learning_rate,
            max_depth = params.max_depth,
            gamma = params.gamma,
            subsample = params.subsample,
            target_col = schema.name
        )
        return model_trainer_config

In [ ]:
import os
from src.stroke_prediction import logger
import pandas as pd
from xgboost import XGBClassifier
import joblib

In [53]:
class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config

    def train_model(self):
        logger.info("Loading training data")
        train_data = pd.read_csv(self.config.train_data_path)
        test_data = pd.read_csv(self.config.test_data_path)

        train_x = train_data.drop(self.config.target_col, axis=1)
        test_x = test_data.drop(self.config.target_col, axis=1)
        train_y = train_data[self.config.target_col]
        test_y = test_data[self.config.target_col]

        lr = XGBClassifier(
            n_estimators=self.config.n_estimators,
            learning_rate=self.config.learning_rate,
            max_depth=self.config.max_depth,
            gamma=self.config.gamma,
            subsample=self.config.subsample,
            random_state=42
        )
        lr.fit(train_x, train_y)

        joblib.dump(lr, os.path.join(self.config.root_dir, self.config.model_name))
        logger.info("Model training completed and model is saved")


In [54]:
try:
    config = ConfiguartionManager()
    model_trainer_config = config.get_model_trainer_config()
    model_trainer = ModelTrainer(model_trainer_config)
    model_trainer.train_model()
except Exception as e:
    logger.exception(e)

2026-05-08 19:55:19,752 - INFO - Loading training data


2026-05-08 19:55:22,070 - INFO - Model training completed and model is saved
